# 01 - Polygon Snapshot Profile

Profiles top-mover snapshot data and full market snapshot quality for scanner readiness.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path('..').resolve()))
from pathlib import Path
import pandas as pd
import plotly.express as px

from src.common import (
    load_env, get_api_keys, load_config, new_pull_id,
    polygon_gainers, polygon_snapshot_all, normalize_polygon_snapshot,
    quality_summary, save_raw_json, save_parquet
)

load_env('../.env')
cfg = load_config('../config/exploration.yaml')
keys = get_api_keys()
assert keys['polygon'], 'POLYGON_API_KEY missing in data-exploration/.env'

cfg.outputs_raw_dir.mkdir(parents=True, exist_ok=True)
cfg.outputs_processed_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
pull_id = new_pull_id('polygon_snapshot')

gainers_payload = polygon_gainers(keys['polygon'], keys['polygon_base'])
all_payload = polygon_snapshot_all(keys['polygon'], keys['polygon_base'])

save_raw_json(gainers_payload, cfg.outputs_raw_dir / f'{pull_id}_gainers.json')
save_raw_json(all_payload, cfg.outputs_raw_dir / f'{pull_id}_all_tickers.json')

source_gainers = '/v2/snapshot/locale/us/markets/stocks/gainers'
source_all = '/v2/snapshot/locale/us/markets/stocks/tickers'

df_gainers = normalize_polygon_snapshot(gainers_payload, source_gainers, pull_id)
df_all = normalize_polygon_snapshot(all_payload, source_all, pull_id)

df_polygon = pd.concat([df_gainers, df_all], ignore_index=True).drop_duplicates(
    subset=['provider', 'symbol', 'ts_utc', 'source_endpoint']
)

save_parquet(df_polygon, cfg.outputs_processed_dir / f'{pull_id}_polygon_snapshot.parquet')
df_polygon.head()


In [ ]:
summary = quality_summary(df_polygon)
summary


In [ ]:
fig1 = px.histogram(df_polygon, x='change_pct', nbins=80, title='Polygon Change % Distribution')
fig1.show()

fig2 = px.histogram(df_polygon, x='volume', nbins=80, title='Polygon Volume Distribution')
fig2.update_xaxes(type='log')
fig2.show()


In [ ]:
null_rates = df_polygon[['provider','symbol','ts_utc','session','last_price','change_pct','volume','prev_close','source_endpoint','pull_id']].isna().mean().sort_values(ascending=False)
null_rates
